# Diffusion-NullFusion Chikusei x4

Multi-Scale Swin Transformer U-Net + DDPM noise prediction in spectral null-space.
Dataset: Chikusei (128 bands, Headwall Hyperspec-VNIR-C, 363-1018nm).


In [ ]:
%%writefile train_diffusion_nullfusion_chikusei.py
"""Diffusion-NullFusion — Chikusei x4 SOTA-beater

Architecture: Multi-Scale Swin Transformer U-Net + DDPM noise prediction
in spectral null-space + Sensor-Adaptive AdaLN.

NaN fixes applied:
  - CG solver: divergence detection, early stopping, float-safe residuals
  - total_loss: per-component NaN guard, returns safe zero on NaN
  - Training loop: skip NaN steps, per-step diagnostics on first NaN
  - UNet inputs clamped to prevent attention overflow
  - First-step full diagnostic dump of all intermediate tensors

Protocol: Wald simulation (Gaussian blur 9x9, sigma=1.2, x4 decimation),
           Chikusei Headwall SRF (128 bands, 363-1018nm).

SOTA targets (Chikusei x4):
  CoFusion      49.14 dB
  SMGU-Net      48.82 dB
  RAMoE         48.10 dB
  DSPNet        ~48.5 dB

Run on Kaggle: python train_diffusion_nullfusion_chikusei.py \
    --root /kaggle/input/chikusei
"""
from __future__ import annotations

import argparse
import atexit
import glob
import json
import math
import os
import random
import signal
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset
from scipy.io import loadmat

try:
    from scipy.ndimage import convolve, uniform_filter
except ImportError:
    from scipy.ndimage import convolve, uniform_filter


# ---------------------------------------------------------------------------
# Chikusei Headwall Hyperspec-VNIR-C SRF (128 bands, 363-1018 nm)
# ---------------------------------------------------------------------------

_CHIKUSEI_128_WL = np.linspace(363.0, 1018.0, 128)
_CHIKUSEI_128_SRF_RAW = np.stack([
    np.exp(-((_CHIKUSEI_128_WL - 620.0) ** 2) / (2 * 80.0 ** 2)),
    np.exp(-((_CHIKUSEI_128_WL - 540.0) ** 2) / (2 * 70.0 ** 2)),
    np.exp(-((_CHIKUSEI_128_WL - 460.0) ** 2) / (2 * 60.0 ** 2)),
], axis=1).astype(np.float32)


def chikusei_srf(bands=128):
    src = _CHIKUSEI_128_SRF_RAW
    if bands != src.shape[0]:
        xs = np.linspace(0.0, 1.0, src.shape[0])
        xd = np.linspace(0.0, 1.0, bands)
        src = np.stack([np.interp(xd, xs, src[:, i]) for i in range(3)], axis=1)
    srf = src.astype(np.float32)
    srf = srf / np.maximum(srf.sum(axis=0, keepdims=True), 1e-8)
    return srf


# ---------------------------------------------------------------------------
# Gaussian blur kernel
# ---------------------------------------------------------------------------

def gaussian_kernel2d(size=9, sigma=1.2):
    ax = np.arange(size, dtype=np.float32) - (size - 1) / 2.0
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-0.5 * (xx ** 2 + yy ** 2) / sigma ** 2)
    return (k / k.sum()).astype(np.float32)


# ---------------------------------------------------------------------------
# Degradation operator (bicubic transpose, not zero-fill)
# ---------------------------------------------------------------------------

class DegradationOp(nn.Module):
    def __init__(self, scale, ksize=9, sigma=1.2):
        super().__init__()
        self.scale = scale
        k = gaussian_kernel2d(ksize, sigma)
        self.register_buffer("k", torch.from_numpy(k)[None, None])

    def forward(self, x):
        C = x.shape[1]
        return F.conv2d(x, self.k.repeat(C, 1, 1, 1), padding=4, groups=C)[
            :, :, :: self.scale, :: self.scale
        ]

    def transpose(self, y, out_hw):
        C = y.shape[1]
        up = F.interpolate(y, size=out_hw, mode="bicubic", align_corners=False)
        return F.conv2d(up, self.k.repeat(C, 1, 1, 1), padding=4, groups=C)


# ---------------------------------------------------------------------------
# Block CG solver with NaN safety and divergence detection
# ---------------------------------------------------------------------------

def block_cg(applyA, rhs, steps, tol=1e-8, max_val=10.0):
    z = tuple(torch.zeros_like(r) for r in rhs)
    r = tuple(rh - a for rh, a in zip(rhs, applyA(z)))
    p = tuple(ri.clone() for ri in r)
    rs = sum((ri * ri).flatten(1).sum(1) for ri in r)
    shape = (rhs[0].shape[0],) + (1,) * (rhs[0].dim() - 1)
    prev_rs = None
    for i in range(steps):
        ap = applyA(p)
        ap = tuple(a.clamp(-max_val, max_val) for a in ap)
        denom = sum((pi * ai).flatten(1).sum(1) for pi, ai in zip(p, ap))
        alpha = (rs / denom.clamp_min(tol)).reshape(*shape)
        z = tuple((zi + alpha * pi).clamp(-max_val, max_val)
                  for zi, pi in zip(z, p))
        r = tuple(ri - alpha * ai for ri, ai in zip(r, ap))
        r = tuple(ri.clamp(-max_val, max_val) for ri in r)
        rs_new = sum((ri * ri).flatten(1).sum(1) for ri in r)
        if prev_rs is not None:
            ratio = rs_new / prev_rs.clamp_min(1e-20)
            if (ratio > 1.0 + 1e-3).any():
                break
        prev_rs = rs_new.clone()
        if (rs_new < tol * tol).all():
            break
        beta = (rs_new / rs.clamp_min(tol)).reshape(*shape)
        p = tuple(ri + beta * pi for ri, pi in zip(r, p))
        rs = rs_new
    return z


# ---------------------------------------------------------------------------
# Combined operator A = [D; R]
# ---------------------------------------------------------------------------

class CombinedOperator(nn.Module):
    def __init__(self, scale, bands, msi_bands, srf_np, ksize=9, sigma=1.2,
                 cg_steps=20, ridge=1e-4):
        super().__init__()
        self.D = DegradationOp(scale, ksize, sigma)
        self.scale = scale
        self.bands = bands
        self.msi_bands = msi_bands
        self.cg_steps = cg_steps
        self.ridge = ridge
        srf = torch.from_numpy(srf_np).float()
        self.register_buffer("srf", srf)

    def R(self, x):
        return torch.einsum("bchw,cm->bmhw", x, self.srf)

    def Rt(self, m):
        return torch.einsum("bmhw,cm->bchw", m, self.srf)

    def forward(self, x):
        return self.D(x), self.R(x)

    def adjoint(self, yH, yM, out_hw):
        return self.D.transpose(yH, out_hw) + self.Rt(yM)

    def apply_gram(self, zH, zM, out_hw):
        Dt = self.D.transpose(zH, out_hw)
        RtM = self.Rt(zM)
        wH = self.D(Dt) + self.D(RtM) + self.ridge * zH
        wM = self.R(Dt) + self.R(RtM) + self.ridge * zM
        return wH, wM

    def pinv(self, yH, yM, out_hw):
        zH, zM = block_cg(
            lambda p: self.apply_gram(p[0], p[1], out_hw), (yH, yM), self.cg_steps
        )
        result = self.adjoint(zH, zM, out_hw)
        return result.clamp(-10.0, 10.0)

    def project_null(self, v):
        with torch.no_grad():
            out_hw = (v.shape[-2], v.shape[-1])
            yH, yM = self.forward(v)
            proj = self.pinv(yH, yM, out_hw)
        result = v - proj
        return result.clamp(-10.0, 10.0)


# ---------------------------------------------------------------------------
# Swin Transformer components
# ---------------------------------------------------------------------------

class WindowAttention(nn.Module):
    def __init__(self, dim, num_heads, window_size=8):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.window_size = window_size

    def forward(self, x):
        B, N, C = x.shape
        H = W = int(math.sqrt(N))
        ws = self.window_size
        x_2d = x.reshape(B, H, W, C)
        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws
        if pad_h or pad_w:
            x_2d = F.pad(x_2d, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = H + pad_h, W + pad_w
        x_win = x_2d.reshape(B, Hp // ws, ws, Wp // ws, ws, C)
        x_win = x_win.permute(0, 1, 3, 2, 4, 5).reshape(-1, ws * ws, C)
        qkv = self.qkv(x_win).reshape(-1, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(1)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(-1, ws * ws, C)
        out = self.proj(out)
        out = out.reshape(B, Hp // ws, Wp // ws, ws, ws, C)
        out = out.permute(0, 1, 3, 2, 4, 5).reshape(B, Hp, Wp, C)
        if pad_h or pad_w:
            out = out[:, :H, :W, :]
        return out.reshape(B, H * W, C)


class AdaLN(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.proj = nn.Linear(cond_dim, dim * 2)

    def forward(self, x, cond):
        gamma, beta = self.proj(cond).unsqueeze(1).chunk(2, dim=-1)
        return self.norm(x) * (1 + gamma) + beta


class SpectralAttention(nn.Module):
    def __init__(self, dim, reduction=4):
        super().__init__()
        hidden = max(dim // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(),
            nn.Linear(hidden, dim), nn.Sigmoid(),
        )

    def forward(self, x):
        gate = self.fc(x.mean(dim=1))
        return x * gate.unsqueeze(1)


class SwinBlock(nn.Module):
    def __init__(self, dim, num_heads, cond_dim, window_size=8, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = AdaLN(dim, cond_dim)
        self.attn = WindowAttention(dim, num_heads, window_size)
        self.norm2 = AdaLN(dim, cond_dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
        )
        self.spec = SpectralAttention(dim)

    def forward(self, x, cond):
        x = x + self.attn(self.norm1(x, cond))
        x = x + self.mlp(self.norm2(x, cond))
        x = self.spec(x)
        return x


class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x, H, W):
        B = x.shape[0]
        x = x.reshape(B, H, W, -1)
        x0 = x[:, 0::2, 0::2]
        x1 = x[:, 1::2, 0::2]
        x2 = x[:, 0::2, 1::2]
        x3 = x[:, 1::2, 1::2]
        x = torch.cat([x0, x1, x2, x3], -1).reshape(B, -1, 4 * x.shape[-1])
        return self.reduction(self.norm(x)), H // 2, W // 2


class PatchExpanding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, 4 * dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, H, W):
        x = self.linear(self.norm(x))
        B = x.shape[0]
        x = x.reshape(B, H, W, 2, 2, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).reshape(B, H * 2, W * 2, -1)
        return x.reshape(B, H * 2 * W * 2, -1), H * 2, W * 2


class ChannelProject(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)


# ---------------------------------------------------------------------------
# Multi-Scale Swin Transformer U-Net (with gradient checkpointing)
# ---------------------------------------------------------------------------

class MultiScaleSwinUNet(nn.Module):
    def __init__(self, in_ch=128, base_dim=48, num_heads=None, cond_dim=64,
                 window_size=8, depths=None, use_checkpoint=False):
        super().__init__()
        if num_heads is None:
            num_heads = [4, 8, 16]
        if depths is None:
            depths = [2, 2, 4, 2, 2]

        dims = [base_dim, base_dim * 2, base_dim * 4]
        self.use_checkpoint = use_checkpoint

        self.input_proj = nn.Linear(in_ch, dims[0])
        self.enc1 = nn.ModuleList([
            SwinBlock(dims[0], num_heads[0], cond_dim, window_size)
            for _ in range(depths[0])
        ])
        self.merge1 = PatchMerging(dims[0])
        self.enc2 = nn.ModuleList([
            SwinBlock(dims[1], num_heads[1], cond_dim, window_size)
            for _ in range(depths[1])
        ])
        self.merge2 = PatchMerging(dims[1])
        self.bottleneck = nn.ModuleList([
            SwinBlock(dims[2], num_heads[2], cond_dim, window_size)
            for _ in range(depths[2])
        ])
        self.expand2 = PatchExpanding(dims[2])
        self.proj_skip2 = ChannelProject(dims[1], dims[2])
        self.proj_dec2 = ChannelProject(dims[2], dims[1])
        self.dec2 = nn.ModuleList([
            SwinBlock(dims[1], num_heads[1], cond_dim, window_size)
            for _ in range(depths[3])
        ])
        self.expand1 = PatchExpanding(dims[1])
        self.proj_skip1 = ChannelProject(dims[0], dims[1])
        self.proj_dec1 = ChannelProject(dims[1], dims[0])
        self.dec1 = nn.ModuleList([
            SwinBlock(dims[0], num_heads[0], cond_dim, window_size)
            for _ in range(depths[4])
        ])
        self.output_proj = nn.Linear(dims[0], in_ch)

    def _run(self, blocks, x, cond):
        for b in blocks:
            if self.use_checkpoint and self.training and x.requires_grad:
                x = checkpoint(b, x, cond, use_reentrant=False)
            else:
                x = b(x, cond)
        return x

    def forward(self, x, cond):
        B, C, H, W = x.shape
        x = x.reshape(B, C, H * W).permute(0, 2, 1)
        x = self.input_proj(x)
        x = self._run(self.enc1, x, cond)
        skip1 = x
        x, H1, W1 = self.merge1(x, H, W)
        x = self._run(self.enc2, x, cond)
        skip2 = x
        x, H2, W2 = self.merge2(x, H1, W1)
        x = self._run(self.bottleneck, x, cond)
        x, H2, W2 = self.expand2(x, H2, W2)
        x = self.proj_dec2(x + self.proj_skip2(skip2))
        x = self._run(self.dec2, x, cond)
        x, H1, W1 = self.expand1(x, H1, W1)
        x = self.proj_dec1(x + self.proj_skip1(skip1))
        x = self._run(self.dec1, x, cond)
        return self.output_proj(x).reshape(B, H, W, -1).permute(0, 3, 1, 2)


# ---------------------------------------------------------------------------
# Cosine diffusion schedule
# ---------------------------------------------------------------------------

class CosineSchedule:
    def __init__(self, T=1000, s=0.008):
        self.T = T
        steps = torch.arange(T + 1, dtype=torch.float64)
        f = torch.cos((steps / T + s) / (1 + s) * math.pi * 0.5) ** 2
        self.alpha_bar = (f / f[0]).float()
        beta = 1 - self.alpha_bar[1:] / self.alpha_bar[:-1]
        self.beta = torch.clamp(beta, max=0.999).float()
        self.alpha = 1 - self.beta

    def to(self, device):
        self.alpha_bar = self.alpha_bar.to(device)
        self.beta = self.beta.to(device)
        self.alpha = self.alpha.to(device)
        return self

    def add_noise(self, x0, noise, t):
        ab = self.alpha_bar[t].reshape(-1, 1, 1, 1)
        return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise

    def ddim_step(self, x_t, eps_pred, t, t_prev):
        ab_t = self.alpha_bar[t].reshape(-1, 1, 1, 1)
        ab_prev = self.alpha_bar[t_prev].reshape(-1, 1, 1, 1)
        x0_pred = (x_t - torch.sqrt(1 - ab_t) * eps_pred) / torch.sqrt(ab_t)
        x0_pred = x0_pred.clamp(0, 1)
        direction = torch.sqrt(1 - ab_prev) * eps_pred
        return torch.sqrt(ab_prev) * x0_pred + direction


# ---------------------------------------------------------------------------
# Sensor embedding (SRF -> conditioning vector)
# ---------------------------------------------------------------------------

class SensorEmbedding(nn.Module):
    def __init__(self, srf_matrix, out_dim=64):
        super().__init__()
        flat = torch.from_numpy(srf_matrix).float().flatten()
        self.register_buffer("srf_flat", flat)
        self.mlp = nn.Sequential(
            nn.Linear(flat.numel(), 128), nn.GELU(), nn.Linear(128, out_dim),
        )

    def forward(self):
        return self.mlp(self.srf_flat.unsqueeze(0)).squeeze(0)


# ---------------------------------------------------------------------------
# Diffusion-NullFusion (Chikusei x4)
# ---------------------------------------------------------------------------

class DiffusionNullFusion(nn.Module):
    def __init__(self, bands=128, msi=3, base_dim=48, scale=4,
                 cond_dim=64, T=1000, num_heads=None, depths=None,
                 window_size=8, use_checkpoint=False, cg_steps=20):
        super().__init__()
        self.bands = bands
        self.scale = scale
        self.T = T

        srf_np = chikusei_srf(bands)
        srf = torch.from_numpy(srf_np).float()
        self.register_buffer("srf", srf)

        self.op = CombinedOperator(
            scale, bands, msi, srf_np, cg_steps=cg_steps, ridge=1e-4,
        )

        self.sensor_embed = SensorEmbedding(srf_np, cond_dim)

        self.schedule = CosineSchedule(T)

        self.cond_proj = nn.Sequential(
            nn.Conv2d(msi + bands, cond_dim, 1),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

        self.time_mlp = nn.Sequential(
            nn.Linear(1, cond_dim), nn.GELU(), nn.Linear(cond_dim, cond_dim),
        )

        self.unet = MultiScaleSwinUNet(
            in_ch=bands, base_dim=base_dim, num_heads=num_heads,
            cond_dim=cond_dim, window_size=window_size, depths=depths,
            use_checkpoint=use_checkpoint,
        )

    def _conditioning(self, yH, yM, H_hr, W_hr):
        with torch.no_grad():
            base = self.op.pinv(yH, yM, (H_hr, W_hr))
        obs = F.interpolate(yH, (H_hr, W_hr), mode="bicubic", align_corners=False)
        cond = self.cond_proj(torch.cat([yM, obs], dim=1))
        cond = cond + self.sensor_embed()
        return cond, base

    def forward(self, x_t, t, cond):
        t_emb = self.time_mlp(t.float().unsqueeze(-1))
        return self.unet(x_t, cond + t_emb)

    @torch.no_grad()
    def inference(self, yH, yM, num_samples=5, ddim_steps=50):
        H_hr, W_hr = yM.shape[-2], yM.shape[-1]
        cond, base = self._conditioning(yH, yM, H_hr, W_hr)

        samples = []
        for _ in range(num_samples):
            x_t = torch.randn(1, self.bands, H_hr, W_hr, device=yH.device)
            ts = torch.linspace(self.T - 1, 0, ddim_steps, dtype=torch.long, device=yH.device)
            for i in range(len(ts) - 1):
                eps = self(x_t, ts[i], cond)
                x_t = self.schedule.ddim_step(x_t, eps, ts[i], ts[i + 1])
            samples.append(x_t)

        avg = torch.mean(torch.stack(samples), dim=0)
        avg_null = self.op.project_null(avg)
        return {"out": (base + avg_null).clamp(0, 1), "base": base, "null": avg_null}


# ---------------------------------------------------------------------------
# Chikusei Dataset (single .mat, 128 bands, patch split)
# ---------------------------------------------------------------------------

class ChikuseiDataset(Dataset):
    """Chikusei dataset loader.

    The Kaggle dataset (mingliu123/chikusei) ships a single .mat file:
        HyperspecVNIR_Chikusei_20140729.mat  (2517 x 2335 x 128)

    We split the full scene into non-overlapping patches with 70/30 train/test.
    MSI is synthesised from HSI using the sensor-specific SRF (Wald protocol).
    """

    def __init__(self, root, split="train", bands=128, scale=4,
                 patch_size=64, max_dim=None):
        self.root = root
        self.split = split
        self.bands = bands
        self.scale = scale
        self.patch_size = patch_size
        self.is_train = split.lower() == "train"
        self.srf = chikusei_srf(bands)
        self.kernel = gaussian_kernel2d(9, 1.2)

        self.cube = self._load_cube(root)
        C, H, W = self.cube.shape
        print(f"[Chikusei] Cube: {C} bands, {H}x{W} pixels")

        self.patches = self._make_patches(H, W)
        print(f"[Chikusei] {split}: {len(self.patches)} patches")

    def _load_cube(self, root):
        search_dirs = [root]
        for parent in [os.path.dirname(root)]:
            if os.path.isdir(parent):
                for d in os.listdir(parent):
                    full = os.path.join(parent, d)
                    if os.path.isdir(full):
                        search_dirs.append(full)
        for sd in search_dirs:
            mat_files = glob.glob(os.path.join(sd, "**", "*.mat"), recursive=True)
            if not mat_files:
                mat_files = glob.glob(os.path.join(sd, "*.mat"))
            if mat_files:
                mat_files.sort(key=lambda f: os.path.getsize(f), reverse=True)
                mat_path = mat_files[0]
                print(f"[Chikusei] Loading: {mat_path}")
                break
        else:
            tree = []
            for d in search_dirs:
                if not os.path.isdir(d):
                    continue
                tree.append(f"--- {d} ---")
                for dirpath, dirnames, filenames in os.walk(d):
                    depth = dirpath.replace(d, "").count(os.sep)
                    if depth > 3:
                        dirnames[:] = []
                        continue
                    tree.append(f"  {dirpath} dirs={sorted(dirnames)[:6]} files={sorted(filenames)[:6]}")
                    if len(tree) > 50:
                        break
            raise FileNotFoundError(
                f"No .mat files found under {root}\n" + "\n".join(tree[:50])
            )

        try:
            data = loadmat(mat_path)
        except NotImplementedError:
            import h5py
            data = {}
            with h5py.File(mat_path, "r") as f:
                for key in f.keys():
                    if not key.startswith("__"):
                        val = np.array(f[key])
                        if getattr(val, "ndim", 0) >= 2:
                            data[key] = val

        for key, val in data.items():
            if str(key).startswith("__") or not hasattr(val, "shape"):
                continue
            arr = np.array(val, dtype=np.float32)
            if arr.ndim != 3 or min(arr.shape) <= 10:
                continue
            if arr.shape[0] > arr.shape[-1]:
                arr = arr.transpose(2, 0, 1)
            if self.bands != arr.shape[0]:
                xs = np.linspace(0.0, 1.0, arr.shape[0])
                xd = np.linspace(0.0, 1.0, self.bands)
                H_, W_ = arr.shape[1], arr.shape[2]
                flat = arr.reshape(arr.shape[0], -1)
                res = np.empty((self.bands, flat.shape[1]), dtype=np.float32)
                for i in range(flat.shape[1]):
                    res[:, i] = np.interp(xd, xs, flat[:, i])
                arr = res.reshape(self.bands, H_, W_)
            arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0)
            if arr.max() > 1.0:
                arr = arr / arr.max()
            arr = np.clip(arr, 0.0, 1.0)
            return arr.astype(np.float32)
        raise ValueError(f"No valid 3D array in {mat_path}")

    def _make_patches(self, H, W):
        p = self.patch_size
        stride = p
        coords = [(y, x) for y in range(0, H - p + 1, stride)
                   for x in range(0, W - p + 1, stride)]
        random.seed(42)
        random.shuffle(coords)
        n_train = int(0.7 * len(coords))
        return coords[:n_train] if self.is_train else coords[n_train:]

    def __len__(self):
        return len(self.patches) * (100 if self.is_train else 1)

    def _simulate(self, gt):
        C, H, W = gt.shape
        blurred = np.empty_like(gt)
        for c in range(C):
            blurred[c] = convolve(gt[c], self.kernel, mode="wrap")
        hr = H // self.scale
        y0 = (H - hr * self.scale) // 2
        x0 = (W - hr * self.scale) // 2
        lr = blurred[:, y0::self.scale, x0::self.scale].astype(np.float32)
        msi = np.einsum("chw,cm->mhw", gt, self.srf).astype(np.float32)
        return lr, np.clip(msi, 0, 1)

    def __getitem__(self, idx):
        y, x = self.patches[idx % len(self.patches)]
        p = self.patch_size
        gt = self.cube[:, y:y + p, x:x + p].copy()
        if self.is_train:
            if random.random() < 0.5:
                gt = gt[:, :, ::-1].copy()
            if random.random() < 0.5:
                gt = gt[:, ::-1, :].copy()
            if random.random() < 0.5:
                k = np.random.randint(1, 4)
                gt = np.rot90(gt, k, axes=(-2, -1)).copy()
        lr, msi = self._simulate(gt)
        return {
            "gt": torch.from_numpy(gt.astype(np.float32)),
            "lr": torch.from_numpy(lr.astype(np.float32)),
            "msi": torch.from_numpy(msi.astype(np.float32)),
            "scene": f"patch_{y}_{x}",
        }


# ---------------------------------------------------------------------------
# Losses (charbonnier + SSIM + SAM + gradient + physics)
# ---------------------------------------------------------------------------

def charbonnier_loss(pred, target, eps=1e-3):
    return torch.sqrt((pred - target) ** 2 + eps ** 2).mean()


def ssim_loss(pred, target, data_range=1.0, size=11, sigma=1.5):
    c = pred.shape[1]
    coords = torch.arange(size, device=pred.device, dtype=pred.dtype) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    win = (g[:, None] @ g[None, :]).expand(c, 1, size, size)
    mu1 = F.conv2d(pred, win, padding=size // 2, groups=c)
    mu2 = F.conv2d(target, win, padding=size // 2, groups=c)
    mu1s, mu2s, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = F.conv2d(pred * pred, win, padding=size // 2, groups=c) - mu1s
    s2 = F.conv2d(target * target, win, padding=size // 2, groups=c) - mu2s
    s12 = F.conv2d(pred * target, win, padding=size // 2, groups=c) - mu12
    c1 = (0.01 * data_range) ** 2
    c2 = (0.03 * data_range) ** 2
    ssim_map = ((2 * mu12 + c1) * (2 * s12 + c2)) / ((mu1s + mu2s + c1) * (s1 + s2 + c2))
    return 1.0 - ssim_map.mean()


def sam_loss(pred, target, eps=1e-6):
    p, t = pred.flatten(2), target.flatten(2)
    num = (p * t).sum(dim=1)
    den = p.norm(dim=1) * t.norm(dim=1)
    cos = (num / den.clamp_min(eps)).clamp(-1 + 1e-6, 1 - 1e-6)
    return torch.acos(cos).mean()


def gradient_loss(pred, target):
    dx_p = pred[..., :, 1:] - pred[..., :, :-1]
    dx_t = target[..., :, 1:] - target[..., :, :-1]
    dy_p = pred[..., 1:, :] - pred[..., :-1, :]
    dy_t = target[..., 1:, :] - target[..., :-1, :]
    return F.l1_loss(dx_p, dx_t) + F.l1_loss(dy_p, dy_t)


def diffusion_loss(model, x0, cond, schedule, min_snr_gamma=5.0):
    B = x0.shape[0]
    t = torch.randint(0, schedule.T, (B,), device=x0.device)
    noise = torch.randn_like(x0)
    x_t = schedule.add_noise(x0, noise, t)
    x_t = x_t.clamp(-5.0, 5.0)
    pred = model(x_t, t, cond)
    per_sample = F.mse_loss(pred, noise, reduction="none").flatten(1).mean(1)
    ab = schedule.alpha_bar[t].clamp(1e-5, 1 - 1e-5)
    snr = ab / (1 - ab)
    w = (snr.clamp(max=min_snr_gamma) / snr).detach()
    return (per_sample * w).mean()


def physics_loss(pred, yH, yM, op):
    lr_pred = op.D(pred)
    msi_pred = op.R(pred)
    return F.mse_loss(lr_pred, yH) + F.mse_loss(msi_pred, yM)


def total_loss(model, gt, yH, yM, schedule,
               w_char=1.0, w_ssim=0.5, w_sam=0.05, w_grad=0.2,
               w_noise=1.0, w_phys=0.1, min_snr_gamma=5.0):
    H_hr, W_hr = yM.shape[-2], yM.shape[-1]
    gt32 = gt.float()
    yH32 = yH.float()
    yM32 = yM.float()

    with torch.no_grad():
        cond, base = model._conditioning(yH32, yM32, H_hr, W_hr)
        x0 = model.op.project_null(gt32 - base)

    x0 = x0.clamp(-5.0, 5.0)

    l_noise = diffusion_loss(model, x0.detach(), cond.detach(), schedule, min_snr_gamma)

    pred = (base + x0).detach().clamp(-1.0, 2.0).requires_grad_(True)
    l_char = charbonnier_loss(pred, gt32)
    l_ssim = ssim_loss(pred.clamp(0, 1), gt32)
    l_sam = sam_loss(pred, gt32)
    l_grad = gradient_loss(pred, gt32)
    l_phys = physics_loss(pred, yH32, yM32, model.op)

    losses = {
        "noise": l_noise, "phys": l_phys,
        "char": l_char, "ssim": l_ssim,
        "sam": l_sam, "grad": l_grad,
    }

    total = torch.tensor(0.0, device=gt.device, requires_grad=True)
    logs = {}
    ok = True
    for name, val in losses.items():
        if not math.isfinite(val.item()):
            print(f"  [NaN] {name} = {val.item()}")
            ok = False
        else:
            w = {"noise": w_noise, "phys": w_phys, "char": w_char,
                 "ssim": w_ssim, "sam": w_sam, "grad": w_grad}[name]
            total = total + w * val
        logs[name] = val.item()

    if not ok:
        total = torch.tensor(0.0, device=gt.device, requires_grad=True)
        for p in model.parameters():
            if p.grad is not None:
                p.grad.zero_()
        logs["nan_flag"] = 1.0
    else:
        logs["nan_flag"] = 0.0

    return total, logs


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------

def psnr_np(pred, gt):
    mse = np.mean((pred - gt) ** 2)
    return 100.0 if mse < 1e-12 else -10 * np.log10(mse)


def sam_np(pred, gt):
    p = pred.reshape(pred.shape[0], -1)
    g = gt.reshape(gt.shape[0], -1)
    p = p / (np.linalg.norm(p, axis=0, keepdims=True) + 1e-8)
    g = g / (np.linalg.norm(g, axis=0, keepdims=True) + 1e-8)
    return np.mean(np.arccos(np.clip((p * g).sum(0), -1, 1))) * 180 / math.pi


def ssim_np(pred, gt):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    mu1 = uniform_filter(pred, 3, mode="reflect")
    mu2 = uniform_filter(gt, 3, mode="reflect")
    s12 = uniform_filter(pred * gt, 3, mode="reflect") - mu1 * mu2
    s1 = uniform_filter(pred ** 2, 3, mode="reflect") - mu1 ** 2
    s2 = uniform_filter(gt ** 2, 3, mode="reflect") - mu2 ** 2
    return np.mean(((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) /
                   ((mu1 ** 2 + mu2 ** 2 + C1) * (s1 + s2 + C2) + 1e-8))


def ergas_np(pred, gt, scale=4):
    C = pred.shape[0]
    e = sum(((pred - gt) ** 2)[c].mean() / (gt[c].mean() ** 2 + 1e-8)
            for c in range(C))
    return math.sqrt(e / C) * 100 * scale


# ---------------------------------------------------------------------------
# EMA
# ---------------------------------------------------------------------------

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point and k in self.shadow:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)

    def apply(self, model):
        model.load_state_dict(self.shadow, strict=False)

    def restore(self, model):
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}


# ---------------------------------------------------------------------------
# Kaggle-safe checkpoint manager
# ---------------------------------------------------------------------------

class KaggleCheckpoint:
    def __init__(self, save_dir, device="cuda"):
        self.save_dir = save_dir
        self.device = device
        self.ckpt_path = os.path.join(save_dir, "resume.pth")
        self.best_path = os.path.join(save_dir, "best.pth")
        self._t0 = time.time()
        self._limit = 11.5 * 3600
        self._saved = False
        self._objects = {}
        self._register_signal()

    def _register_signal(self):
        if sys.platform == "win32":
            return
        original = signal.getsignal(signal.SIGTERM)

        def handler(signum, frame):
            if self._saved:
                return
            print(f"\n[KAGGLE] Signal {signum} -- force saving...")
            try:
                self.save(**self._objects, force=True)
                self._saved = True
            except Exception as e:
                print(f"[KAGGLE] Save failed: {e}")
            if callable(original):
                original(signum, frame)

        try:
            signal.signal(signal.SIGTERM, handler)
        except (OSError, ValueError):
            pass

    def time_remaining(self):
        return max(0.0, self._limit - (time.time() - self._t0))

    def should_save(self, step, every=500):
        if step % every == 0:
            return True
        if self.time_remaining() < 1800 and step % 200 == 0:
            return True
        return False

    def save(self, model, ema, opt, scheduler, epoch, best_psnr, best_epoch,
             force=False):
        self._objects = dict(
            model=model, ema=ema, opt=opt, scheduler=scheduler,
            epoch=epoch, best_psnr=best_psnr, best_epoch=best_epoch,
        )
        tmp = self.ckpt_path + ".tmp"
        torch.save({
            "model": model.state_dict(),
            "ema": ema.shadow,
            "opt": opt.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "best_psnr": best_psnr,
            "best_epoch": best_epoch,
            "wall_time": time.time() - self._t0,
        }, tmp)
        os.replace(tmp, self.ckpt_path)

    def save_best(self, model, opt, ema, scheduler, epoch, best_psnr, best_epoch,
                  val_metrics):
        self.save(model, ema, opt, scheduler, epoch, best_psnr, best_epoch)
        torch.save({
            "model": model.state_dict(),
            "val": val_metrics,
            "epoch": epoch,
        }, self.best_path)
        print(f"[ckpt] Best saved: PSNR={val_metrics['psnr']:.4f}")

    def load(self, model, ema, opt, scheduler):
        if not os.path.exists(self.ckpt_path):
            return 1, float("-inf"), 0
        ck = torch.load(self.ckpt_path, map_location=self.device, weights_only=False)
        model.load_state_dict(ck["model"])
        ema.shadow = {k: v.to(self.device) for k, v in ck["ema"].items()}
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["scheduler"])
        epoch = ck["epoch"]
        best_psnr = ck.get("best_psnr", float("-inf"))
        best_epoch = ck.get("best_epoch", 0)
        wall = ck.get("wall_time", 0)
        print(f"[ckpt] Resumed: epoch {epoch}, best PSNR={best_psnr:.4f}, "
              f"wall={wall / 3600:.1f}h")
        return epoch + 1, best_psnr, best_epoch


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", type=str, required=True,
                        help="Chikusei dataset root")
    parser.add_argument("--scale", type=int, default=4)
    parser.add_argument("--bands", type=int, default=128)
    parser.add_argument("--patch", type=int, default=64)
    parser.add_argument("--batch_size", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=5000)
    parser.add_argument("--steps_per_epoch", type=int, default=200)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--grad_accum", type=int, default=4)
    parser.add_argument("--eval_every", type=int, default=20)
    parser.add_argument("--time_budget_h", type=float, default=8.5)
    parser.add_argument("--T", type=int, default=1000)
    parser.add_argument("--ddim_steps", type=int, default=50)
    parser.add_argument("--num_samples", type=int, default=5)
    parser.add_argument("--base_dim", type=int, default=48)
    parser.add_argument("--cond_dim", type=int, default=64)
    parser.add_argument("--cg_steps", type=int, default=20)
    parser.add_argument("--save_dir", type=str, default="/kaggle/working/diffusion_nullfusion")
    args = parser.parse_args()

    os.makedirs(args.save_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"Memory: {mem:.1f} GB")

    use_checkpoint = True
    if torch.cuda.is_available():
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        if mem < 8:
            args.patch = 48
            args.batch_size = 1
            args.base_dim = 32
        elif mem < 12:
            args.patch = 56
            args.batch_size = 2
            args.base_dim = 40
        else:
            args.patch = 64
            args.batch_size = 2
            args.base_dim = 48

    model = DiffusionNullFusion(
        bands=args.bands, msi=3, base_dim=args.base_dim, scale=args.scale,
        cond_dim=args.cond_dim, T=args.T,
        use_checkpoint=use_checkpoint, cg_steps=args.cg_steps,
    ).to(device)

    nparams = sum(p.numel() for p in model.parameters())
    print(f"Model: Diffusion-NullFusion Chikusei -- {nparams / 1e6:.2f}M params")
    print(f"Config: patch={args.patch} batch={args.batch_size} dim={args.base_dim}")

    schedule = model.schedule.to(device)

    train_ds = ChikuseiDataset(args.root, "train", args.bands, args.scale, args.patch)
    test_ds = ChikuseiDataset(args.root, "test", args.bands, args.scale, args.patch)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4,
                            betas=(0.9, 0.999))
    ema = EMA(model, 0.999)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=200, T_mult=2, eta_min=1e-6,
    )
    use_amp = False
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    ckpt = KaggleCheckpoint(args.save_dir, device=str(device))
    start_epoch, best_psnr, best_epoch = ckpt.load(model, ema, opt, scheduler)

    T0 = time.time()
    LIMIT = args.time_budget_h * 3600

    targets = {
        "CoFusion (2026)": 49.14,
        "DSPNet (2023)": 48.50,
        "SMGU-Net (2025)": 48.82,
        "RAMoE (2024)": 48.10,
    }
    our_prev_best = 48.10

    print(f"\nTraining: {args.epochs} epochs, {args.time_budget_h}h budget")
    print(f"Diffusion T={args.T}, DDIM steps={args.ddim_steps}, samples={args.num_samples}")
    print(f"Effective batch: {args.batch_size * args.grad_accum}, Steps/epoch: {args.steps_per_epoch}")
    print("-" * 70)

    # ---- Smoke test with real data ----
    print("[SMOKE] Running 1-step loss on real data batch...")
    model.train()
    _batch = [train_ds[random.randrange(len(train_ds))] for _ in range(2)]
    _sgt = torch.stack([b["gt"] for b in _batch], 0).to(device)
    _slr = torch.stack([b["lr"] for b in _batch], 0).to(device)
    _smsi = torch.stack([b["msi"] for b in _batch], 0).to(device)
    print(f"  gt: [{_sgt.min():.4f}, {_sgt.max():.4f}] shape={_sgt.shape}")
    print(f"  lr: [{_slr.min():.4f}, {_slr.max():.4f}] shape={_slr.shape}")
    print(f"  msi: [{_smsi.min():.4f}, {_smsi.max():.4f}] shape={_smsi.shape}")

    with torch.no_grad():
        cond, base = model._conditioning(_slr, _smsi, _smsi.shape[-2], _smsi.shape[-1])
        print(f"  base: [{base.min():.4f}, {base.max():.4f}] nan={base.isnan().any()}")
        x0 = model.op.project_null(_sgt - base)
        print(f"  x0: [{x0.min():.4f}, {x0.max():.4f}] nan={x0.isnan().any()}")

    _sloss, _slogs = total_loss(model, _sgt, _slr, _smsi, schedule)
    print(f"[SMOKE] loss={_sloss.item():.6f} finite={math.isfinite(_sloss.item())}")
    for k, v in _slogs.items():
        if k != "nan_flag":
            print(f"  {k}: {v:.6f} finite={math.isfinite(v)}")
    nan_flag = _slogs.get("nan_flag", 0.0)
    if nan_flag > 0:
        print("[SMOKE] NaN detected in loss components on real data! Continuing with NaN-safe mode.")
    else:
        print("[SMOKE] PASSED.\n")

    nan_steps_total = 0

    for epoch in range(start_epoch, args.epochs + 1):
        if (time.time() - T0) > LIMIT:
            print(f"\n[TIME BUDGET at epoch {epoch}]")
            ckpt.save(model, ema, opt, scheduler, epoch - 1, best_psnr, best_epoch,
                      force=True)
            break

        model.train()
        total = 0
        t0 = time.time()
        opt.zero_grad()
        nan_steps = 0
        _debug = epoch <= 3

        for step in range(args.steps_per_epoch):
            batch = [train_ds[random.randrange(len(train_ds))]
                     for _ in range(args.batch_size)]
            gt = torch.stack([b["gt"] for b in batch], 0).to(device)
            yH = torch.stack([b["lr"] for b in batch], 0).to(device)
            yM = torch.stack([b["msi"] for b in batch], 0).to(device)

            loss, logs = total_loss(model, gt, yH, yM, schedule)

            is_nan = not math.isfinite(loss.item()) or logs.get("nan_flag", 0) > 0

            if is_nan:
                nan_steps += 1
                if _debug and nan_steps <= 3:
                    print(f"  [NaN STEP] epoch={epoch} step={step}")
                    print(f"    gt: [{gt.min():.4f}, {gt.max():.4f}] nan={gt.isnan().any()}")
                    print(f"    lr: [{yH.min():.4f}, {yH.max():.4f}] nan={yH.isnan().any()}")
                    print(f"    msi: [{yM.min():.4f}, {yM.max():.4f}] nan={yM.isnan().any()}")
                    for k, v in logs.items():
                        if k != "nan_flag":
                            print(f"    {k}: {v:.6f}")
                opt.zero_grad()
            else:
                (loss / args.grad_accum).backward()

            if (step + 1) % args.grad_accum == 0 and not is_nan:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad()
                ema.update(model)

            if not is_nan:
                total += loss.item()

            if ckpt.should_save(epoch * args.steps_per_epoch + step):
                ckpt.save(model, ema, opt, scheduler, epoch, best_psnr, best_epoch)

        scheduler.step()
        dt = time.time() - t0
        valid_steps = max(1, args.steps_per_epoch - nan_steps)
        avg = total / valid_steps
        nan_steps_total += nan_steps

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:5d}/{args.epochs} | Loss {avg:.6f} | "
                  f"LR {scheduler.get_last_lr()[0]:.2e} | {dt:.1f}s"
                  + (f" | NaN steps: {nan_steps}/{args.steps_per_epoch}" if nan_steps else ""))

        if epoch % args.eval_every == 0 or epoch == args.epochs:
            raw_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            ema.apply(model)
            model.eval()
            ps_list, ss_list, sa_list, er_list = [], [], [], []

            with torch.no_grad():
                for i in range(min(len(test_ds), 50)):
                    item = test_ds[i]
                    out = model.inference(
                        item["lr"].unsqueeze(0).to(device),
                        item["msi"].unsqueeze(0).to(device),
                        num_samples=1, ddim_steps=20,
                    )
                    pred_np = out["out"][0].cpu().numpy().clip(0, 1)
                    gt_np = item["gt"].numpy()
                    ps_list.append(psnr_np(pred_np, gt_np))
                    ss_list.append(ssim_np(pred_np, gt_np))
                    sa_list.append(sam_np(pred_np, gt_np))
                    er_list.append(ergas_np(pred_np, gt_np, args.scale))

            mp = float(np.mean(ps_list))
            ms_ = float(np.mean(ss_list))
            ma = float(np.mean(sa_list))
            me = float(np.mean(er_list))

            marker = ""
            if mp > best_psnr:
                best_psnr = mp
                best_epoch = epoch
                ckpt.save_best(model, opt, ema, scheduler, epoch,
                               best_psnr, best_epoch,
                               {"psnr": mp, "ssim": ms_, "sam": ma, "ergas": me})
                marker = " [BEST]"

            print(f"  >>> Test@{epoch}: PSNR {mp:.4f} | SSIM {ms_:.4f} | "
                  f"SAM {ma:.3f} | ERGAS {me:.3f}{marker}")
            for name, target in targets.items():
                d = mp - target
                m_ = " <<< BEAT" if d > 0 else ""
                print(f"    vs {name:20s}: delta={d:+.2f} dB{m_}")

            model.load_state_dict(raw_state)
            model.train()

            ckpt.save(model, ema, opt, scheduler, epoch, best_psnr, best_epoch,
                      force=True)

    print(f"\nTotal NaN steps across training: {nan_steps_total}")

    print("\n" + "=" * 70)
    print("FINAL EVALUATION: 5 samples x 50 DDIM steps, FULL test set")
    print("=" * 70)

    best_ckpt = os.path.join(args.save_dir, "best.pth")
    if os.path.exists(best_ckpt):
        ck = torch.load(best_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
    model.eval()
    ps_list, ss_list, sa_list, er_list = [], [], [], []

    with torch.no_grad():
        for i in range(len(test_ds)):
            item = test_ds[i]
            out = model.inference(
                item["lr"].unsqueeze(0).to(device),
                item["msi"].unsqueeze(0).to(device),
                num_samples=args.num_samples,
                ddim_steps=args.ddim_steps,
            )
            pred_np = out["out"][0].cpu().numpy().clip(0, 1)
            gt_np = item["gt"].numpy()
            ps_list.append(psnr_np(pred_np, gt_np))
            ss_list.append(ssim_np(pred_np, gt_np))
            sa_list.append(sam_np(pred_np, gt_np))
            er_list.append(ergas_np(pred_np, gt_np, args.scale))

    final = {
        "psnr": float(np.mean(ps_list)),
        "ssim": float(np.mean(ss_list)),
        "sam": float(np.mean(sa_list)),
        "ergas": float(np.mean(er_list)),
    }
    print(f"\nFINAL (epoch {best_epoch}): PSNR {final['psnr']:.4f} | "
          f"SSIM {final['ssim']:.4f} | SAM {final['sam']:.3f} | "
          f"ERGAS {final['ergas']:.3f}")

    for name, target in {**targets, "Stretch (50 dB)": 50.0}.items():
        d = final["psnr"] - target
        m_ = " <<< BEAT" if d > 0 else ""
        print(f"  {name:25s} Target {target:6.2f} | Ours delta={d:+.2f} dB{m_}")

    results = {
        "dataset": "Chikusei",
        "bands": args.bands,
        "protocol": "Wald x4, Chikusei Headwall SRF, Gaussian 9x9 sigma=1.2",
        "params_M": nparams / 1e6,
        "best_epoch": best_epoch,
        "T": args.T,
        "ddim_steps": args.ddim_steps,
        "num_samples": args.num_samples,
        "targets": targets,
        "final": final,
    }
    out_path = os.path.join(args.save_dir, "results.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
!pip install --quiet scipy Pillow h5py 2>/dev/null
import os; os.makedirs('/kaggle/working/diffusion_nullfusion', exist_ok=True)
print('Setup done')


In [ ]:
import os, glob
candidates = [
    '/kaggle/input/chikusei',
    '/kaggle/input/mingliu123/chikusei',
    '/kaggle/input/mingliu123-chikusei',
]
root = None
for c in candidates:
    if os.path.isdir(c):
        mats = glob.glob(os.path.join(c, '**', '*.mat'), recursive=True)
        if not mats: mats = glob.glob(os.path.join(c, '*.mat'))
        if mats:
            root = c
            print(f'Found .mat at {c}: {[os.path.basename(m) for m in mats[:3]]}')
            break
if root is None:
    for d in ['/kaggle/input']:
        if not os.path.isdir(d): continue
        for dirpath, dirs, files in os.walk(d):
            for f in files:
                fp = os.path.join(dirpath, f)
                if f.endswith('.mat') and os.path.getsize(fp) > 1e8:
                    root = d
                    print(f'Found large .mat: {fp}')
                    break
            if root: break
        if root: break
if root is None:
    raise FileNotFoundError('Chikusei .mat not found!')
print(f'Using root: {root}')
!python train_diffusion_nullfusion_chikusei.py --root {root}
